In [ ]:
import pandas as pd, os
pd.set_option('display.width', 200, 'display.max_columns', 50)

ROOT = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
ser  = pd.read_csv(f'{ROOT}/train_series.csv')

print('columns:', list(ser.columns))
print('series:', len(ser), '| studies:', ser.StudyInstanceUID.nunique())
print()
print(ser.groupby(['Anatomical_Plane', 'Fluid_Sensitive']).size().unstack(fill_value=0))
print()

# how many studies have a non-FS series in each plane
ns = ser[ser.Fluid_Sensitive == 0]
print('studies with a non-FS series, by plane:')
print(ns.groupby('Anatomical_Plane').StudyInstanceUID.nunique())
print('studies with >=1 non-FS series anywhere:', ns.StudyInstanceUID.nunique(), '/ 4407')
print()

# what are they? look for any descriptive column
for c in ser.columns:
    if ser[c].dtype == object and c not in ('StudyInstanceUID', 'SeriesInstanceUID'):
        print(f'--- {c} (non-FS only) ---')
        print(ns[c].value_counts().head(25))
        print()

In [ ]:
print(ser.groupby(['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']).size())

In [ ]:
import os, cv2, numpy as np, pandas as pd, pydicom
from multiprocessing import Pool
from tqdm.auto import tqdm

ROOT   = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
SPLIT  = 'train'
K, S   = 24, 192
PLANES = ['Sagittal', 'Coronal']      # non-fluid-sensitive versions of these two
OUT    = f'/kaggle/working/{SPLIT}_nfs'
CHUNK  = 256
os.makedirs(OUT, exist_ok=True)
cv2.setNumThreads(0)

ser  = pd.read_csv(f'{ROOT}/{SPLIT}_series.csv')
ser  = ser[ser.Fluid_Sensitive == 0]                      # the only change from before
CAND = {k: g.SeriesInstanceUID.tolist() for k, g in ser.groupby(['StudyInstanceUID','Anatomical_Plane'])}
UIDS = pd.read_csv(f'{ROOT}/{SPLIT}.csv').StudyInstanceUID.tolist()   # same order as knee-shards-all

def sort_key(ds):
    try:
        iop = np.array(ds.ImageOrientationPatient, float)
        n = np.cross(iop[:3], iop[3:])
        return float(np.array(ds.ImagePositionPatient, float) @ n)
    except Exception:
        return float(getattr(ds, 'InstanceNumber', 0) or 0)

def load_series(path):
    files = [os.path.join(path, f) for f in os.listdir(path)]
    keys  = [sort_key(pydicom.dcmread(f, stop_before_pixels=True)) for f in files]
    order = np.argsort(keys)
    idx   = np.linspace(0, len(files) - 1, K).round().astype(int)
    cache, sl = {}, []
    for i in idx:
        f = files[order[i]]
        if f not in cache:
            img = pydicom.dcmread(f).pixel_array.astype(np.float32)
            if img.ndim == 3: img = img[img.shape[0] // 2]
            cache[f] = img
        sl.append(cache[f])
    vol = np.stack(sl)
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1) * 255
    out = np.zeros((K, S, S), np.uint8)
    for j, im in enumerate(vol):
        h, w = im.shape; m = max(h, w)
        pad = np.zeros((m, m), np.float32)
        pad[(m-h)//2:(m-h)//2+h, (m-w)//2:(m-w)//2+w] = im
        out[j] = cv2.resize(pad, (S, S), interpolation=cv2.INTER_AREA).astype(np.uint8)
    return out

def process_study(uid):
    arr  = np.zeros((len(PLANES), K, S, S), np.uint8)
    mask = np.zeros(len(PLANES), np.uint8)
    for p, plane in enumerate(PLANES):
        sids = CAND.get((uid, plane), [])
        if not sids: continue
        sids = sorted(sids, key=lambda s: -len(os.listdir(f'{ROOT}/{SPLIT}_series/{uid}/{s}')))
        for s in sids:
            try:
                arr[p] = load_series(f'{ROOT}/{SPLIT}_series/{uid}/{s}'); mask[p] = 1; break
            except Exception as e:
                print('fail', uid, s, repr(e)[:80], flush=True)
    return uid, arr, mask

chunks = [UIDS[i:i+CHUNK] for i in range(0, len(UIDS), CHUNK)]
with Pool(os.cpu_count(), maxtasksperchild=64) as pool:
    for ci, ch in enumerate(tqdm(chunks)):
        fn = f'{OUT}/shard_{ci:04d}.npz'
        if os.path.exists(fn): continue
        res = pool.map(process_study, ch, chunksize=4)
        np.savez(fn, x=np.stack([r[1] for r in res]),
                     m=np.stack([r[2] for r in res]),
                     uid=np.array([r[0] for r in res]))
        print(f'shard {ci} done', flush=True)

d = np.load(f'{OUT}/shard_0000.npz')
print(d['x'].shape, d['m'].mean(0), d['uid'][0] == UIDS[0])